# Mathematical Self-Improvement Agent — Minimal Demo

This notebook walks through the full PPO self-play loop:
1. Load the dual-task model
2. Run **one** rollout (question generation → solve → reward)
3. Show the reward breakdown
4. Run a mini PPO update

Designed to run on a free Colab T4 GPU with a small model.

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────────
!pip install -q torch transformers peft sympy matplotlib seaborn

In [ ]:
import sys, os
# If running in Colab, clone the repo first:
# !git clone https://github.com/<your-repo>/Finetune_qwen.git && cd Finetune_qwen
sys.path.insert(0, os.path.abspath(".."))  # adjust if needed

## 1. Load Model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Use a small model for the demo; replace with your dual-task checkpoint.
MODEL_ID = "Qwen/Qwen2.5-Math-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

policy = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
print(f"Loaded {MODEL_ID} on {policy.device}")

## 2. Initialize Components

In [ ]:
from src.rl.reward_calculator import RewardCalculator
from src.rl.value_network import ValueHead
from src.rl.math_environment import MathEnvironment

reward_calculator = RewardCalculator()

value = ValueHead(MODEL_ID, freeze_backbone=True).to(policy.device)

env = MathEnvironment(
    policy_model=policy,
    value_model=value,
    tokenizer=tokenizer,
    reward_calculator=reward_calculator,
    max_question_tokens=80,   # small for demo speed
    max_solution_tokens=200,
    temperature=0.7,
    top_p=0.9,
)
print("Environment ready.")

## 3. Single Rollout

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

trajectory = env.rollout_trajectory()

print("\n" + "=" * 70)
print("INSTRUCTION:")
print(trajectory.metadata["instruction"])
print("\nGENERATED QUESTION:")
print(trajectory.metadata["generated_question"])
print("\nGENERATED SOLUTION:")
print(trajectory.metadata["generated_solution"])
print("\nREWARD BREAKDOWN:")
rb = trajectory.metadata["reward_breakdown"]
print(f"  Combined:          {rb['combined_score']:.3f}")
print(f"  Question quality:  {rb['question_metrics']['overall_score']:.3f}")
print(f"  Solution quality:  {rb['solution_metrics']['overall_score']:.3f}")
print(f"  Episode length:    {len(trajectory)} tokens")
print("=" * 70)

## 4. Collect Multiple Rollouts and Run One PPO Update

In [ ]:
from src.rl.rollout_buffer import RolloutBuffer
from src.rl.ppo_trainer import PPOTrainer

# Collect a small batch
N_ROLLOUTS = 5
trajectories = env.collect_rollouts(num_trajectories=N_ROLLOUTS)

buffer = RolloutBuffer(gamma=1.0, gae_lambda=0.95)
for traj in trajectories:
    buffer.add_trajectory(traj)

stats = buffer.get_stats()
print(f"Buffer: {stats['num_trajectories']} trajectories, "
      f"mean reward = {stats['mean_episode_reward']:.3f}")

In [ ]:
trainer = PPOTrainer(
    policy_model=policy,
    value_model=value,
    tokenizer=tokenizer,
    learning_rate=1e-5,
    ppo_epochs=2,
    batch_size=8,
)

metrics = trainer.train_step(buffer)
print("PPO update metrics:", metrics)

## 5. Reward Visualization

In [ ]:
import matplotlib.pyplot as plt

rewards = [t.total_reward for t in trajectories]

plt.figure(figsize=(8, 3))
plt.bar(range(len(rewards)), rewards, color="steelblue", alpha=0.8)
plt.axhline(sum(rewards) / len(rewards), color="red", linestyle="--",
            label=f"Mean = {sum(rewards)/len(rewards):.3f}")
plt.xlabel("Trajectory")
plt.ylabel("Total Reward")
plt.title(f"Reward per Trajectory (N={N_ROLLOUTS})")
plt.legend()
plt.tight_layout()
plt.show()